# Repeated K-Fold — real / synthetic / mixed (mean ± 95% CI)

Runs the same 5-fold CV suite `len(SEEDS)` times with different random seeds.
Each seed produces one OOF F1 value per experiment. Results are aggregated as
**mean ± std** and **95% CI** (t-distribution, `df = n_seeds − 1`).

| Pose experiment | Calibration |
|---|---|
| Real pipeline | — |
| Synthetic (Kimodo) | Per-fold (avoids leakage) |
| Mixed | Per-fold (avoids leakage) |


In [1]:
from pathlib import Path
import dataclasses
import sys
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    root for root in candidate_roots
    if (root / 'pose_module').exists() and (root / 'evaluation').exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from evaluation.classifiers import (
    WindowedDatasetConfig,
    build_classifier_capture_table,
    build_windowed_multimodal_dataset
)
from evaluation.classifiers import (
    EXPERIMENT_SPECS,
    ModelConfig,
    SplitConfig,
    TrainingConfig
)
from pose_module.robot_emotions.metadata import get_protocol_info

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 260)
PROJECT_ROOT

PosixPath('/home/henriquesouza/POSE2IMU-Framework')

## Configuration

- `SEEDS` — random seeds used for repeated K-fold. Each produces one complete 5-fold run.
- All other parameters are identical to `classifiers_pose_experiments.ipynb`.


In [2]:
REAL_OUTPUT_ROOT   = PROJECT_ROOT / 'output' / 'robot_emotions_virtual_imu'
SYNTHETIC_MANIFEST = PROJECT_ROOT / 'output' / 'robot_emotions_kimodo_imu'  / 'virtual_imu_manifest.jsonl'
MIXED_MANIFEST     = PROJECT_ROOT / 'output' / 'robot_emotions_mixed_imu'   / 'mixed_virtual_imu_manifest.jsonl'
ANCHOR_CATALOG     = PROJECT_ROOT / 'output' / 'robot_emotions_kimodo_anchors' / 'kimodo_anchor_catalog.jsonl'

IMU_FEATURE_MODE     = 'acc_euler'
CALIBRATION_FRACTION = 0.1

EXCLUDED_EMOTIONS = {'Anger', 'Fear'}
EXCLUDED_STIMULI  = {}

# Seeds for repeated K-fold — each produces one independent OOF run.
SEEDS = [42, 123, 456, 789, 1234, 19, 2074, 9973, 7581, 6412]

DATASET_CONFIG = WindowedDatasetConfig(
    window_size=81,
    overlap=0.5,
    synthetic_variant='raw',
    imu_feature_mode=IMU_FEATURE_MODE,
    selected_sensors=None,
    max_windows_per_capture=None,
    random_state=42,
)
SPLIT_CONFIG = SplitConfig(n_splits=5, random_state=42)
MODEL_CONFIG  = ModelConfig(hidden_dim=128, dropout=0.1, trunk_blocks=2, modality_dropout_p=0.1)
TRAINING_CONFIG = TrainingConfig(
    batch_size=64, max_epochs=10, learning_rate=2e-3, weight_decay=1e-4,
    device='cuda',
    domain_loss_weight=0.1, flat_tag_loss_weight=0.2, emotion_loss_weight=0.75,
    modality_loss_weight=0.05, stimulus_loss_weight=0.25, use_cb_focal=True,
    show_progress=False
    )

REQUESTED_ORDER = [
    'vision_only', 'imu_only_r2r', 'imu_only_s2r', 'imu_only_mixed2r',
    'vision_imu_r2r', 'vision_imu_s2r', 'vision_imu_mixed2r',
]

print('SEEDS:', SEEDS)
print('n_splits:', SPLIT_CONFIG.n_splits)
print('Total runs per experiment:', len(SEEDS), 'x', SPLIT_CONFIG.n_splits, 'folds')

SEEDS: [42, 123, 456, 789, 1234, 19, 2074, 9973, 7581, 6412]
n_splits: 5
Total runs per experiment: 10 x 5 folds


## Helper functions — data loading & capture tables

Identical to `classifiers_pose_experiments.ipynb`.


In [3]:
def _load_real_manifest_index(real_output_root: Path) -> pd.DataFrame:
    path = real_output_root / 'virtual_imu_manifest.jsonl'
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        e = json.loads(line)
        clip_dir = str(Path(e['artifacts']['virtual_imu_npz_path']).parent.parent.parent.resolve())
        rows.append({
            'clip_id':    str(e['clip_id']),
            'domain':     str(e['domain']),
            'user_id':    int(e['user_id']),
            'tag_number': int(e['tag_number']),
            'take_id':    e.get('take_id'),
            'clip_dir':   clip_dir,
        })
    return pd.DataFrame(rows).set_index('clip_id')


def _load_anchor_catalog_index(anchor_catalog_path: Path) -> dict[str, tuple[float, float]]:
    index = {}
    for line in anchor_catalog_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        e = json.loads(line)
        wid = e.get('window_id') or e.get('prompt_id')
        if wid and 'window' in e and e['window']:
            index[wid] = (float(e['window']['start_sec']), float(e['window']['end_sec']))
    return index


def _build_synthetic_capture_table(manifest_path, real_index, anchor_index, *, pose_kind_filter=None):
    rows = []
    for line in manifest_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        e = json.loads(line)
        if e.get('status') not in ('ok', 'warning'):
            continue
        pose_kind = e.get('pose_kind', 'synthetic')
        if pose_kind_filter is not None and pose_kind != pose_kind_filter:
            continue

        if pose_kind == 'real':
            artifacts      = e.get('artifacts', {})
            pose3d_npz     = artifacts.get('pose3d_npz_path')
            virtual_imu_npz = artifacts.get('virtual_imu_npz_path')
            uncalibrated_npz = artifacts.get('virtual_imu_geometric_aligned_npz_path')
            ref_clip_id    = str(e['clip_id'])
        else:
            va             = e.get('virtual_imu_artifacts', {})
            pose3d_npz     = va.get('pose3d_npz_path')
            if pose3d_npz is None:
                pose3d_npz = (e.get('constraint_summary') or {}).get('pose3d_source_path')
            virtual_imu_npz  = va.get('virtual_imu_npz_path')
            uncalibrated_npz = va.get('virtual_imu_npz_path')
            ref_clip_id      = str(e.get('reference_clip_id') or e.get('clip_id'))

        if pose3d_npz is None or virtual_imu_npz is None:
            continue
        if not Path(pose3d_npz).exists() or not Path(virtual_imu_npz).exists():
            continue
        if ref_clip_id not in real_index.index:
            continue

        real_row = real_index.loc[ref_clip_id]
        real_imu_ref_path = str(Path(real_row['clip_dir']) / 'imu.npz')
        protocol = get_protocol_info(real_row['domain'], real_row['tag_number']) or {}
        window_id = e.get('window_id')
        real_imu_time_range_sec = anchor_index.get(window_id) if window_id else None
        sample_id = str(e.get('sample_id') or e.get('prompt_id') or e['clip_id'])

        rows.append({
            'clip_id':                            sample_id,
            'reference_clip_id':                  ref_clip_id,
            'domain':                             real_row['domain'],
            'user_id':                            real_row['user_id'],
            'tag_number':                         real_row['tag_number'],
            'take_id':                            real_row['take_id'],
            'emotion':                            str(protocol.get('emotion', '')),
            'modality':                           str(protocol.get('modality', '')),
            'stimulus':                           str(protocol.get('stimulus', 'None')),
            'status':                             str(e.get('status', 'ok')),
            'pose3d_npz_path':                    pose3d_npz,
            'virtual_imu_npz_path':               virtual_imu_npz,
            'virtual_imu_uncalibrated_npz_path':  uncalibrated_npz,
            'real_imu_reference_npz_path':        real_imu_ref_path,
            'virtual_imu_frame_aligned_npz_path': None,
            'clip_dir':                           real_row['clip_dir'],
            'pose_kind':                          pose_kind,
            'quality_report':                     {},
            'real_imu_time_range_sec':            real_imu_time_range_sec,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df['subject_group']        = df.apply(lambda r: f"{r['domain']}_user_{int(r['user_id']):02d}", axis=1)
    df['flat_tag']             = df.apply(lambda r: f"{r['emotion']}|{r['modality']}|{r['stimulus']}", axis=1)
    df['frame_aligned_available'] = False
    print(f'  real_imu_time_range_sec preenchido: {df["real_imu_time_range_sec"].notna().sum()}/{len(df)}')
    print(f'  virtual_imu_uncalibrated_npz_path preenchido: {df["virtual_imu_uncalibrated_npz_path"].notna().sum()}/{len(df)}')
    return df.reset_index(drop=True)


def _apply_exclusions(df: pd.DataFrame, excluded_emotions: set[str], excluded_stimuli: set[str]) -> pd.DataFrame:
    if df.empty:
        return df
    mask = pd.Series(False, index=df.index)
    if excluded_emotions:
        em = df['emotion'].isin(excluded_emotions)
        if em.any():
            print(f'  Excluindo {int(em.sum())} capturas com emoção em {excluded_emotions}')
        mask |= em
    if excluded_stimuli:
        sm = df['stimulus'].isin(excluded_stimuli)
        if sm.any():
            print(f'  Excluindo {int(sm.sum())} capturas com estímulo em {excluded_stimuli}')
        mask |= sm
    return df.loc[~mask].reset_index(drop=True)


def _sort_suite_summary(summary_df: pd.DataFrame) -> pd.DataFrame:
    order_map = {name: idx for idx, name in enumerate(REQUESTED_ORDER)}
    df = summary_df.copy()
    df['_sort_key'] = df['experiment_name'].astype(str).map(order_map).fillna(len(REQUESTED_ORDER))
    return df.sort_values('_sort_key', kind='stable').drop(columns=['_sort_key']).reset_index(drop=True)


REAL_INDEX   = _load_real_manifest_index(REAL_OUTPUT_ROOT)
ANCHOR_INDEX = _load_anchor_catalog_index(ANCHOR_CATALOG)
print(f'Real manifest index: {len(REAL_INDEX)} clips')
print(f'Anchor catalog index: {len(ANCHOR_INDEX)} windows')

Real manifest index: 89 clips
Anchor catalog index: 3725 windows


## Helper functions — per-fold calibration

Identical to `classifiers_pose_experiments.ipynb`.


In [4]:
from evaluation.classifiers.experiments import (
    build_subject_group_splits,
    run_single_experiment,
    _aggregate_oof_report,
    _build_oof_summary,
)
from evaluation.classifiers.metrics import (
    build_scored_class_ids,
    build_support_report,
    compute_domain_gap_summary,
    suite_results_frame,
    PRIMARY_HEADS,
)
from pose_module.processing.imu_calibration import (
    build_calibration_reference_matrix,
    calibrate_virtual_imu_sequence,
)
from evaluation.classifiers.features import build_imu_feature_tensor
from evaluation.tsne import segment_signal_windows


def _build_train_real_imu_matrix(dataset_bundle: dict, train_capture_ids: set[str], *, calibration_fraction: float = 1.0) -> np.ndarray:
    capture_table = dataset_bundle['capture_table']
    train_rows = capture_table[capture_table['clip_id'].isin(train_capture_ids)]
    seen_paths: set[str] = set()
    clip_npz_paths = []
    for _, row in train_rows.iterrows():
        ref_path = row.get('real_imu_reference_npz_path')
        if not ref_path or not Path(ref_path).exists():
            candidate = Path(row['clip_dir']) / 'imu.npz'
            if not candidate.exists():
                continue
            ref_path = str(candidate)
        if ref_path in seen_paths:
            continue
        seen_paths.add(ref_path)
        clip_npz_paths.append(ref_path)
    if not clip_npz_paths:
        raise RuntimeError('No training clip with a valid imu.npz found.')
    return build_calibration_reference_matrix(
        clip_npz_paths=clip_npz_paths,
        target_sensor_names=dataset_bundle['selected_sensors'],
        signal_mode='acc',
        calibration_fraction=calibration_fraction,
    )


def _recalibrate_synthetic_windows( dataset_bundle: dict, train_indices: np.ndarray) -> np.ndarray:
    import tempfile, os
    from pose_module.interfaces import VirtualIMUSequence

    metadata         = dataset_bundle['metadata']
    capture_table    = dataset_bundle['capture_table']
    imu_synthetic    = np.array(dataset_bundle['imu_synthetic_windows'], dtype=np.float32)
    imu_feature_mode = dataset_bundle['imu_feature_mode']
    selected_sensors = dataset_bundle['selected_sensors']
    cfg              = dataset_bundle['config']

    train_capture_ids = set(metadata.iloc[train_indices]['capture_id'].astype(str).tolist())
    real_matrix = _build_train_real_imu_matrix(
        dataset_bundle, train_capture_ids, calibration_fraction=CALIBRATION_FRACTION
    )

    tmp = tempfile.NamedTemporaryFile(suffix='.npz', delete=False)
    tmp.close()
    try:
        np.savez(tmp.name,
                 acc=real_matrix.reshape(real_matrix.shape[0], len(selected_sensors), -1),
                 sensor_names=np.array(selected_sensors))
        cap_idx = capture_table.set_index('clip_id')

        for capture_id, window_indices in metadata.groupby('capture_id').groups.items():
            capture_id = str(capture_id)
            if capture_id not in cap_idx.index:
                continue
            cap_row = cap_idx.loc[capture_id]
            uncalibrated_path = cap_row.get('virtual_imu_uncalibrated_npz_path')
            if not uncalibrated_path or pd.isna(uncalibrated_path) or not Path(uncalibrated_path).exists():
                continue

            with np.load(uncalibrated_path, allow_pickle=True) as payload:
                raw_acc          = np.asarray(payload['acc'],            dtype=np.float32)
                raw_gyro         = np.asarray(payload['gyro'],           dtype=np.float32)
                timestamps_sec   = np.asarray(payload['timestamps_sec'], dtype=np.float32)
                sensor_names_raw = [str(v) for v in np.asarray(payload['sensor_names']).tolist()]

            sel_indices = [sensor_names_raw.index(s) for s in selected_sensors if s in sensor_names_raw]
            if len(sel_indices) != len(selected_sensors):
                continue
            acc_sel  = raw_acc[:,  sel_indices, :]
            gyro_sel = raw_gyro[:, sel_indices, :]

            seq = VirtualIMUSequence(
                clip_id=capture_id, fps=None, sensor_names=selected_sensors,
                acc=acc_sel, gyro=gyro_sel, timestamps_sec=timestamps_sec, source='uncalibrated',
            )
            result = calibrate_virtual_imu_sequence(seq, real_imu_reference_path=tmp.name, signal_mode='acc')
            calibrated_acc = np.asarray(result['virtual_imu_sequence'].acc, dtype=np.float32)

            imu_features = build_imu_feature_tensor(
                calibrated_acc, gyro_sel, timestamps_sec, feature_mode=imu_feature_mode
            )
            window_bundle = segment_signal_windows(
                imu_features['values'],
                window_type='n_samples',
                window_size=int(cfg.window_size),
                stride_or_overlap_mode='overlap',
                overlap=float(cfg.overlap),
            )
            new_windows    = np.asarray(window_bundle['windows'], dtype=np.float32)
            idx_array      = np.asarray(window_indices, dtype=np.int64)
            n_replace      = min(len(idx_array), new_windows.shape[0])
            imu_synthetic[idx_array[:n_replace]] = new_windows[:n_replace]
    finally:
        os.unlink(tmp.name)

    return imu_synthetic

## Repeated K-Fold runner

`_run_experiment_for_seed` runs one complete K-fold CV with a single seed.
`_run_repeated_kfold` loops over `SEEDS` and collects per-seed OOF results.

Each seed creates its own `SplitConfig(random_state=seed)` via `dataclasses.replace`,
leaving every other hyperparameter unchanged.


In [5]:
def _run_experiment_for_seed(label: str, captures_df: pd.DataFrame, seed: int, *, use_fold_calibration: bool = False) -> dict:
    split_cfg   = dataclasses.replace(SPLIT_CONFIG,   random_state=seed)
    dataset_cfg = dataclasses.replace(DATASET_CONFIG, random_state=seed)

    dataset_bundle = build_windowed_multimodal_dataset(REAL_OUTPUT_ROOT, config=dataset_cfg, captures_df=captures_df)
    experiment_names = list(EXPERIMENT_SPECS.keys())

    if not use_fold_calibration:
        splits = build_subject_group_splits(dataset_bundle['metadata'], config=split_cfg)
        support_report = build_support_report(
            dataset_bundle['metadata'],
            dataset_bundle['label_encoders'],
            group_column=str(split_cfg.group_column),
            min_subject_groups=int(split_cfg.min_subject_groups_per_class),
        )
        scored_class_ids = build_scored_class_ids(support_report, head_names=PRIMARY_HEADS)

        results = []
        for split in splits:
            split_ctx = {**split, 'num_splits': len(splits)}
            for experiment_name in experiment_names:
                fold_id  = int(split['split_id'])
                result = run_single_experiment(
                    dataset_bundle,
                    experiment_name=experiment_name,
                    split=split_ctx,
                    model_config=MODEL_CONFIG,
                    training_config=TRAINING_CONFIG,
                    scored_class_ids=scored_class_ids,
                )
                results.append(result)

        oof_reports = {
            exp: _aggregate_oof_report(
                [r for r in results if r['experiment_name'] == exp],
                label_encoders=dataset_bundle['label_encoders'],
                scored_class_ids=scored_class_ids,
            )
            for exp in experiment_names
        }
        oof_summary = _build_oof_summary(
            oof_reports,
            support_report=support_report,
            primary_head=str(split_cfg.primary_head),
        )
        suite_result = {
            'results':            results,
            'results_frame':      suite_results_frame(results),
            'summary':            oof_summary,
            'oof_reports':        oof_reports,
            'support_report':     support_report,
            'domain_gap_summary': compute_domain_gap_summary(oof_summary),
            'splits':             splits,
        }

    else:
        # Synthetic / mixed: recalibrate per fold to avoid leakage.
        splits = build_subject_group_splits(dataset_bundle['metadata'], config=split_cfg)
        support_report = build_support_report(
            dataset_bundle['metadata'],
            dataset_bundle['label_encoders'],
            group_column=str(split_cfg.group_column),
            min_subject_groups=int(split_cfg.min_subject_groups_per_class),
        )
        scored_class_ids = build_scored_class_ids(support_report, head_names=PRIMARY_HEADS)

        results = []
        for split in splits:
            fold_id       = int(split['split_id'])
            train_indices = np.asarray(split['train_indices'], dtype=np.int64)
            split_ctx     = {**split, 'num_splits': len(splits)}

            print(f'    Fold {fold_id+1}/{len(splits)}: recalibrating synth. IMU ...')
            recalibrated = _recalibrate_synthetic_windows(dataset_bundle, train_indices)
            fold_bundle  = {**dataset_bundle, 'imu_synthetic_windows': recalibrated}

            for experiment_name in experiment_names:
                result = run_single_experiment(
                    fold_bundle,
                    experiment_name=experiment_name,
                    split=split_ctx,
                    model_config=MODEL_CONFIG,
                    training_config=TRAINING_CONFIG,
                    scored_class_ids=scored_class_ids,
                )
                results.append(result)

        oof_reports = {
            exp: _aggregate_oof_report(
                [r for r in results if r['experiment_name'] == exp],
                label_encoders=dataset_bundle['label_encoders'],
                scored_class_ids=scored_class_ids,
            )
            for exp in experiment_names
        }
        oof_summary = _build_oof_summary(
            oof_reports,
            support_report=support_report,
            primary_head=str(split_cfg.primary_head),
        )
        suite_result = {
            'results':            results,
            'results_frame':      suite_results_frame(results),
            'summary':            oof_summary,
            'oof_reports':        oof_reports,
            'support_report':     support_report,
            'domain_gap_summary': compute_domain_gap_summary(oof_summary),
            'splits':             splits,
        }

    return {
        'label':          label,
        'seed':           seed,
        'dataset_bundle': dataset_bundle,
        'suite_result':   suite_result,
    }


def _run_repeated_kfold(label: str, captures_df: pd.DataFrame, *, use_fold_calibration: bool = False) -> dict:
    captures_df = _apply_exclusions(captures_df.copy(), EXCLUDED_EMOTIONS, EXCLUDED_STIMULI)
    seed_results = []

    for i, seed in enumerate(SEEDS):
        print(f"\n{'='*60}")
        print(f"  {label}  —  seed {seed}  ({i+1}/{len(SEEDS)})")
        print(f"{'='*60}")
        result = _run_experiment_for_seed(label, captures_df, seed, use_fold_calibration=use_fold_calibration)
        seed_results.append(result)
        short = _sort_suite_summary(result['suite_result']['summary'])
        display(short[['experiment_name', 'global_score_macro_f1_mean', 'global_score_weighted_macro_f1', 'emotion_macro_f1']].head(10))

    print(f"\nFinished: {len(seed_results)} seeds × {SPLIT_CONFIG.n_splits} folds for '{label}'")
    return {'label': label, 'seed_results': seed_results}

## CI aggregation and plotting


In [6]:
CI_METRICS = [
    'emotion_macro_f1',
    'modality_macro_f1',
    'stimulus_macro_f1',
    'global_score_macro_f1_mean',
    'global_score_weighted_macro_f1',
]


def compute_ci_summary(repeated_result: dict, metric: str = 'emotion_macro_f1') -> pd.DataFrame:
    """
    Aggregate per-seed OOF scores into mean ± std and 95% CI.
    Uses t-distribution with df = n_seeds - 1 (appropriate for small n).
    """
    exp_scores: dict[str, list[float]] = {}
    for sr in repeated_result['seed_results']:
        for _, row in sr['suite_result']['summary'].iterrows():
            exp = str(row['experiment_name'])
            val = row.get(metric)
            if val is not None and not pd.isna(val):
                exp_scores.setdefault(exp, []).append(float(val))

    order_map = {name: i for i, name in enumerate(REQUESTED_ORDER)}
    rows = []
    for exp_name, scores in sorted(exp_scores.items(), key=lambda x: order_map.get(x[0], 999)):
        n   = len(scores)
        arr = np.array(scores)
        mean = float(np.mean(arr))
        std  = float(np.std(arr, ddof=1)) if n > 1 else 0.0
        sem  = std / np.sqrt(n) if n > 1 else 0.0
        ci_half = float(scipy_stats.t.ppf(0.975, df=n - 1) * sem) if n > 1 else 0.0
        rows.append({
            'experiment_name': exp_name,
            'mean':            round(mean, 4),
            'std':             round(std,  4),
            'ci95_half':       round(ci_half, 4),
            'formatted':       f'{mean:.3f} ± {ci_half:.3f}',
            'n_seeds':         n,
            '_scores':         scores,
        })
    return pd.DataFrame(rows)


def build_full_ci_table(repeated_result: dict) -> pd.DataFrame:
    """
    Multi-metric CI table for all CI_METRICS.
    Each column shows 'mean ± 95% CI' for one metric.
    """
    base = compute_ci_summary(repeated_result, metric=CI_METRICS[0])
    result = base[['experiment_name', 'n_seeds', 'formatted']].rename(
        columns={'formatted': CI_METRICS[0]}
    )
    for metric in CI_METRICS[1:]:
        df = compute_ci_summary(repeated_result, metric=metric)[['experiment_name', 'formatted']]
        df = df.rename(columns={'formatted': metric})
        result = result.merge(df, on='experiment_name', how='left')
    return result


def plot_ci_bar(repeated_results: list[dict], metric: str = 'emotion_macro_f1') -> None:
    """Grouped bar chart with 95% CI error bars — one group per classifier."""
    n_groups = len(REQUESTED_ORDER)
    n_exp    = len(repeated_results)
    bar_w    = 0.8 / n_exp
    x        = np.arange(n_groups)
    colors   = ['#2196F3', '#FF5722', '#4CAF50']

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, rr in enumerate(repeated_results):
        ci_df = compute_ci_summary(rr, metric=metric).set_index('experiment_name')
        means    = [ci_df.loc[e, 'mean']      if e in ci_df.index else 0.0 for e in REQUESTED_ORDER]
        ci_halfs = [ci_df.loc[e, 'ci95_half'] if e in ci_df.index else 0.0 for e in REQUESTED_ORDER]
        offset   = (i - n_exp / 2 + 0.5) * bar_w
        ax.bar(
            x + offset, means, bar_w * 0.9,
            label=rr['label'], color=colors[i % len(colors)], alpha=0.8,
            yerr=ci_halfs, capsize=4, error_kw={'linewidth': 1.5},
        )

    n_seeds = len(repeated_results[0]['seed_results'])
    ax.set_xticks(x)
    ax.set_xticklabels(REQUESTED_ORDER, rotation=30, ha='right')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.set_title(f'{metric}  —  mean ± 95% CI  ({n_seeds} seeds × {SPLIT_CONFIG.n_splits} folds)')
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_seed_boxplot(repeated_results: list[dict], metric: str = 'emotion_macro_f1') -> None:
    """Boxplot of per-seed OOF scores — one panel per pose experiment."""
    fig, axes = plt.subplots(1, len(repeated_results), figsize=(6 * len(repeated_results), 5), sharey=True)
    if len(repeated_results) == 1:
        axes = [axes]

    for ax, rr in zip(axes, repeated_results):
        ci_df  = compute_ci_summary(rr, metric=metric).set_index('experiment_name')
        data   = [ci_df.loc[e, '_scores'] for e in REQUESTED_ORDER if e in ci_df.index]
        labels = [e for e in REQUESTED_ORDER if e in ci_df.index]
        ax.boxplot(data, tick_labels=labels, vert=True)
        ax.set_title(rr['label'])
        ax.set_xticklabels(labels, rotation=45, ha='right')
        ax.set_ylabel(metric)
        ax.set_ylim(0, 1)

    fig.suptitle(f'{metric}  —  distribution per seed', y=1.02)
    plt.tight_layout()
    plt.show()

## Build capture tables (once — reused across all seeds)


In [7]:
print('--- Real pipeline captures ---')
CAPTURES_REAL = build_classifier_capture_table(REAL_OUTPUT_ROOT)
print(f'{len(CAPTURES_REAL)} clips')

print('\n--- Synthetic (Kimodo) captures ---')
CAPTURES_SYNTHETIC = _build_synthetic_capture_table(SYNTHETIC_MANIFEST, REAL_INDEX, ANCHOR_INDEX, pose_kind_filter='synthetic')
print(f'{len(CAPTURES_SYNTHETIC)} clips')

print('\n--- Mixed captures ---')
CAPTURES_MIXED = _build_synthetic_capture_table(MIXED_MANIFEST, REAL_INDEX, ANCHOR_INDEX, pose_kind_filter=None)
print(f'{len(CAPTURES_MIXED)} clips')

--- Real pipeline captures ---
80 clips

--- Synthetic (Kimodo) captures ---
  real_imu_time_range_sec preenchido: 3725/3725
  virtual_imu_uncalibrated_npz_path preenchido: 3725/3725
3725 clips

--- Mixed captures ---
  real_imu_time_range_sec preenchido: 3722/3811
  virtual_imu_uncalibrated_npz_path preenchido: 3811/3811
3811 clips


---
## Experiment 1 — Real pipeline  ×  N seeds


In [ ]:
REPEATED_REAL = _run_repeated_kfold('real_pose', CAPTURES_REAL, use_fold_calibration=True)

  Excluindo 4 capturas com emoção em {'Fear', 'Anger'}

  real_pose  —  seed 42  (1/10)
    Fold 1/5: recalibrating synth. IMU ...


---
## Experiment 2 — Synthetic (Kimodo)  ×  N seeds


In [ ]:
REPEATED_SYNTHETIC = _run_repeated_kfold( 'synthetic_pose', CAPTURES_SYNTHETIC, use_fold_calibration=True)

---
## Experiment 3 — Mixed  ×  N seeds


In [ ]:
REPEATED_MIXED = _run_repeated_kfold( 'mixed_pose', CAPTURES_MIXED, use_fold_calibration=True)

---
## Results — mean ± 95% CI per experiment

Each cell shows `mean ± half-width` of the 95% confidence interval
(t-distribution, `df = n_seeds − 1 = 4`).


In [ ]:
for rr in (REPEATED_REAL, REPEATED_SYNTHETIC, REPEATED_MIXED):
    display(Markdown(f"### {rr['label']}"))
    display(build_full_ci_table(rr))

## Combined table — all three experiments side-by-side


In [ ]:
metric = 'emotion_macro_f1'

parts = []
for rr in (REPEATED_REAL, REPEATED_SYNTHETIC, REPEATED_MIXED):
    df = compute_ci_summary(rr, metric=metric)[['experiment_name', 'mean', 'std', 'ci95_half', 'formatted', 'n_seeds']]
    df = df.rename(columns={
        'mean':      f'{rr["label"]}_mean',
        'std':       f'{rr["label"]}_std',
        'ci95_half': f'{rr["label"]}_ci95',
        'formatted': rr['label'],
        'n_seeds':   'n_seeds',
    })
    parts.append(df)

combined = parts[0]
for part in parts[1:]:
    combined = combined.merge(
        part.drop(columns=['n_seeds'], errors='ignore'),
        on='experiment_name', how='outer'
    )

display(combined[['experiment_name', 'real_pose', 'synthetic_pose', 'mixed_pose']])

## Bar chart — mean ± 95% CI


In [ ]:
for metric in CI_METRICS:
    plot_ci_bar([REPEATED_REAL, REPEATED_SYNTHETIC, REPEATED_MIXED], metric=metric)

## Boxplots — distribution per seed


In [ ]:
for metric in ['emotion_macro_f1', 'global_score_macro_f1_mean', 'global_score_weighted_macro_f1']:
    plot_seed_boxplot([REPEATED_REAL, REPEATED_SYNTHETIC, REPEATED_MIXED], metric=metric)

## Per-seed raw scores


In [ ]:
metric = 'emotion_macro_f1'

for rr in (REPEATED_REAL, REPEATED_SYNTHETIC, REPEATED_MIXED):
    display(Markdown(f"### {rr['label']} — per-seed OOF `{metric}`"))
    rows = []
    for sr in rr['seed_results']:
        seed    = sr['seed']
        summary = sr['suite_result']['summary']
        for _, row in _sort_suite_summary(summary).iterrows():
            val = row.get(metric)
            rows.append({'seed': seed, 'experiment_name': row['experiment_name'], metric: val})
    pivot = pd.DataFrame(rows).pivot(index='experiment_name', columns='seed', values=metric)
    pivot.columns = [f'seed_{c}' for c in pivot.columns]
    pivot['mean'] = pivot.mean(axis=1).round(4)
    pivot['std']  = pivot.std(axis=1, ddof=1).round(4)
    display(pivot)